<a href="https://colab.research.google.com/github/cristianzucconi2-web/iatoarts/blob/main/iatoarts3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- 1. SETUP ---
!pip install mido
import random
import mido
from mido import Message, MidiFile, MidiTrack

# --- 2. CONFIGURAZIONE "GRAND OPERA" ---
SCALA = [60, 62, 63, 65, 67, 68, 70, 72] # Do Minore
BASSO_ROOTS = [36, 43, 48, 39] # Note fondamentali per il basso
POPOLAZIONE_SIZE = 100
GENERAZIONI = 120
BLOCCHI = 8 # Per arrivare a circa 2 minuti

# --- 3. FITNESS AVANZATA (MUSICALITÀ) ---
def calcola_fitness_opera(individuo):
    piano = individuo[0]
    basso = individuo[1]
    voto = 0

    for i in range(len(piano)):
        # 1. Armonia Verticale (Il basso deve sostenere il piano)
        # Se il piano suona una nota che è nella triade del basso, bonus enorme
        if (piano[i] - basso[i]) % 12 in [0, 3, 7, 10]: voto += 60

        # 2. Melodia "Cantabile" (Salti piccoli)
        if i < len(piano) - 1:
            salto = abs(piano[i] - piano[i+1])
            if 1 <= salto <= 2: voto += 50 # Gradi congiunti
            if salto > 12: voto -= 150     # No salti impossibili
            if salto == 0: voto -= 20      # Evita troppe note identiche

        # 3. Posizione delle note (Il basso deve stare basso, il piano medio)
        if not (60 <= piano[i] <= 84): voto -= 50
        if not (36 <= basso[i] <= 48): voto -= 50

    return voto

# --- 4. MOTORE EVOLUTIVO ---
def genera_sezione():
    popolazione = []
    for _ in range(POPOLAZIONE_SIZE):
        p = [random.choice(SCALA) for _ in range(32)]
        b = [random.choice(BASSO_ROOTS) for _ in range(32)]
        popolazione.append([p, b])

    for g in range(GENERAZIONI):
        popolazione = sorted(popolazione, key=calcola_fitness_opera, reverse=True)
        migliori = popolazione[:20]
        nuova = migliori.copy()
        while len(nuova) < POPOLAZIONE_SIZE:
            p1, p2 = random.choice(migliori), random.choice(migliori)
            taglio = random.randint(1, 31)
            f = [p1[0][:taglio] + p2[0][taglio:], p1[1][:taglio] + p2[1][taglio:]]
            if random.random() < 0.2: f[0][random.randint(0, 31)] = random.choice(SCALA)
            nuova.append(f)
        popolazione = nuova
    return popolazione[0]

# --- 5. COMPOSIZIONE ---
piano_master, basso_master, archi_master = [], [], []

print("Componendo l'opera multitraccia...")
for b in range(BLOCCHI):
    sez = genera_sezione()
    piano_master.extend(sez[0])
    basso_master.extend(sez[1])
    # Gli archi seguono il basso ma con note lunghe
    archi_master.extend(sez[1])
    print(f"Sezione {b+1} completata.")

# --- 6. GENERAZIONE MIDI PROFESSIONALE ---
mid = MidiFile()
t_piano = MidiTrack()
t_basso = MidiTrack()
t_archi = MidiTrack()
t_drums = MidiTrack()
mid.tracks.extend([t_piano, t_basso, t_archi, t_drums])

# Assegnazione Strumenti (Program Change)
t_piano.append(Message('program_change', program=0, time=0))   # Acoustic Piano
t_basso.append(Message('program_change', program=32, time=0))  # Acoustic Bass
t_archi.append(Message('program_change', program=49, time=0))  # String Ensemble
t_drums.append(Message('program_change', program=0, time=0))   # Standard Drums

def add_note(track, note, duration, vel=80, chan=0):
    track.append(Message('note_on', note=note, velocity=vel, time=0, channel=chan))
    track.append(Message('note_off', note=note, velocity=vel, time=duration, channel=chan))

for i in range(len(piano_master)):
    # PIANO: Fa note veloci (ottavi) - 240 tick
    add_note(t_piano, piano_master[i], 240, vel=90)
    add_note(t_piano, random.choice(SCALA), 240, vel=70) # Aggiunge un abbellimento veloce

    # BASSO: Note lunghe (quarti) - 480 tick
    add_note(t_basso, basso_master[i], 480, vel=100)

    # ARCHI: Note lunghissime (ogni 4 battiti) per fare atmosfera
    if i % 4 == 0:
        add_note(t_archi, archi_master[i] + 12, 1920, vel=60) # Ottava sopra il basso

    # DRUMS: Beat più complesso
    if i % 2 == 0:
        add_note(t_drums, 36, 240, chan=9, vel=100) # Kick
    else:
        add_note(t_drums, 38, 240, chan=9, vel=80)  # Snare
        add_note(t_drums, 42, 240, chan=9, vel=60)  # Hi-hat

mid.save('grand_opera_ai.mid')
print("\nOPERA PRONTA! Scarica 'grand_opera_ai.mid'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.0 MB/s eta 0:00:00
Composizione in corso (6 blocchi)...
Blocco 1 completato!
Blocco 2 completato!
Blocco 3 completato!
Blocco 4 completato!
Blocco 5 completato!
Blocco 6 completato!

Fatto! Scarica 'composizione_finale_ai.mid' e mettilo su GarageBand.
